## Tarea 3
Programa Experto en Inteligencia Artificial con Python  
ML3006 - Web Mining en Python  
Estudiante: Natalia Bonilla Villalobos.  
**Web Scrapping con Selenium**

<a id="menu"></a>

# Menú
- [Ejercicio 1](#ej1)
- [Ejercicio 2](#ej2)

In [1]:
import numpy as np
import requests
import json
import re
import time
import pandas as pd
from selenium import webdriver
from selenium.webdriver.firefox.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.common.by import By
from io import StringIO

import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import OneHotEncoder, StandardScaler

<a id="ej1"></a>
# Ejercicio 1

[50 puntos] **Artesanos Car Club** es una tienda de vehículos. Para este ejercicio extraeremos información del enlace que se encuentra presionando [aquí](https://artesanoscarclub.com/vehiculos-usados/). Obtenga el **url** y usando **Selenium** construya una tabla de datos con todos los vehículos disponibles en el sitio web:


- a) A modo de prueba, extraiga el nombre, el precio, el año, el kilometraje, el motor y el tipo de combustible de un único vehículo para verificar que el código funciona correctamente.
- b) Obtenga el elemento con el enlace del siguiente conjunto de vehículos.
- c) Realice un ciclo para obtener la información de todos los vehículos disponibles, con los mismos datos que se solicitan en el primer inciso.
- d) Con los resultados anteriores genere un DataFrame.
- e) Exporte los datos como `vehiculos.csv`.

[↑ Volver al Menú](#menu)

### Navegar y extraer datos

In [2]:
driver = webdriver.Firefox()
driver.get('https://artesanoscarclub.com/vehiculos-usados/')

In [3]:
wait = WebDriverWait(driver, 60)
wait.until(EC.presence_of_element_located((By.ID, "vehiculos-loop")))

<selenium.webdriver.remote.webelement.WebElement (session="ffbd1a49-9c66-429d-a8e5-ad63c59e9fc6", element="9daba0ee-b2a2-4d22-a0b9-7a5338c8db22")>

In [4]:
div_container = driver.find_elements(By.XPATH, "//div[contains(@class, 'elementor elementor-1046 e-loop-item')]")
print(len(div_container))
print(div_container[0])
print(type(div_container[0]))

31
<selenium.webdriver.remote.webelement.WebElement (session="ffbd1a49-9c66-429d-a8e5-ad63c59e9fc6", element="2211b0f9-91b3-4616-b489-41cbf2fcc8f7")>
<class 'selenium.webdriver.remote.webelement.WebElement'>


## Parte 1
- a) A modo de prueba, extraiga el nombre, el precio, el año, el kilometraje, el motor y el tipo de combustible de un único vehículo para verificar que el código funciona correctamente.

In [5]:
vehicle = div_container[0]

In [6]:
h2_list = vehicle.find_elements(By.TAG_NAME, "h2")
n=0
for h2 in h2_list:
    print(h2.text, n)
    n += 1

NISSAN QASHQAI 1.3L GLORY – 4X2 – 2025. 0
US$ 1
35.000 2


### Nombre

In [7]:
name = vehicle.find_element(By.TAG_NAME, "h2").text
name

'NISSAN QASHQAI 1.3L GLORY – 4X2 – 2025.'

### Precio

In [8]:
price = vehicle.find_elements(By.TAG_NAME, "h2")[2].text.replace(".", "")
price

'35000'

### Año, Kilometraje, Combustible y Motor

In [9]:
span_list = vehicle.find_elements(By.CLASS_NAME, "elementor-icon-list-text")
n = 0
for span in span_list:
    print(n, span.text)
    n += 1

0 3629
1 2025
2 35 Km
3 Gasolina
4 1298 CC
5 4x2 tracción delantera


In [10]:
year = vehicle.find_elements(By.CLASS_NAME, "elementor-icon-list-text")[1].text
year

'2025'

In [11]:
mileage = vehicle.find_elements(By.CLASS_NAME, "elementor-icon-list-text")[2].text.replace(" KM", "").replace(" Km", "").replace(".", "")
mileage

'35'

In [12]:
fuel = vehicle.find_elements(By.CLASS_NAME, "elementor-icon-list-text")[3].text
fuel

'Gasolina'

In [13]:
engine = vehicle.find_elements(By.CLASS_NAME, "elementor-icon-list-text")[4].text.replace(" CC", "")
engine

'1298'

## Parte 2
- b) Obtenga el elemento con el enlace del siguiente conjunto de vehículos.

In [14]:
button = driver.find_element(By.CLASS_NAME, "next")
button.click()

In [15]:
driver.quit()

## Parte 3
- c) Realice un ciclo para obtener la información de todos los vehículos disponibles, con los mismos datos que se solicitan en el primer inciso.
- d) Con los resultados anteriores genere un DataFrame.

In [16]:
def scrape_vehicles():
    data = []
    driver = webdriver.Firefox()
    try:
        driver.get('https://artesanoscarclub.com/vehiculos-usados/')
        
        wait = WebDriverWait(driver, 60)
        
        while True:
            wait.until(EC.presence_of_element_located((By.ID, "vehiculos-loop")))
            time.sleep(5)
            
            div_container = driver.find_elements(By.XPATH, "//div[contains(@class, 'elementor elementor-1046 e-loop-item')]")

            for vehicle in div_container:
                name = vehicle.find_element(By.TAG_NAME, "h2").text
                if not name.strip():
                    continue
                price = vehicle.find_elements(By.TAG_NAME, "h2")[2].text.replace(".", "")
                year = vehicle.find_elements(By.CLASS_NAME, "elementor-icon-list-text")[1].text
                mileage = vehicle.find_elements(By.CLASS_NAME, "elementor-icon-list-text")[2].text.replace(" KM", "").replace(" Km", "").replace(".", "")
                fuel = vehicle.find_elements(By.CLASS_NAME, "elementor-icon-list-text")[3].text
                engine = vehicle.find_elements(By.CLASS_NAME, "elementor-icon-list-text")[4].text.replace(" CC", "")
                
                data.append({
                    "name": name,
                    "price": price,
                    "year": year,
                    "mileage": mileage,
                    "fuel": fuel,
                    "engine": engine
                })
                
            try:
                next_button = driver.find_element(By.CLASS_NAME, "next")
                next_button.click()
                
                wait.until(EC.staleness_of(div_container[0]))
            except:
                break
        
    except Exception as e:
        print("ERROR:", e)
        
    finally:
        driver.quit()
    
    return pd.DataFrame(data)

In [17]:
data = scrape_vehicles()
data

,name,price,year,mileage,fuel,engine
0,NISSAN QASHQAI 1.3L GLORY – 4X2 – 2025.,35000,2025,35,Gasolina,1298
1,LEXUS GX 460 V8 – 2019.,45000,2019,78900,Gasolina,4.608
2,LEXUS LX 570 V8 – 2019.,67000,2019,50900,Gasolina,5.663
3,BMW X1 – 18d – sDRIVE – 2018.,22500,2018,72800,Diésel,1955
4,MERCEDES BENZ E 300 – 2017.,38000,2017,69500,Gasolina,1.991
...,...,...,...,...,...,...
215,MERCEDES BENZ GL 350d AMG – 4matic – 2015.,37000,2015,101700,Diésel,2987
216,MERCEDES BENZ CLS 53 AMG 4matic + – 2019.,95000,2019,20300,Gasolina,3000
217,PORSCHE TAYCAN ELÉCTRICO – 2021.,72000,2021,33000,Eléctrico,0
218,MERCEDES BENZ C 180 2015.,22000,2015,108500,Gasolina,1595


## Part 3
- e) Exporte los datos como `vehiculos.csv`.

In [18]:
data.to_csv("vehiculos.csv", index=False, encoding="utf-8-sig")

<a id="ej2"></a>
# Ejercicio 2

[50 puntos] Automatizar una búsqueda en https://crautos.com/autosusados/index.cfm y construir un DataFrame con los resultados. Tomando en cuenta lo anterior realice lo siguiente: 

- a) Abra la página de autos usados y desplácese hasta el final del formulario de filtros. 
- b) Seleccione **Año desde = 2018 y Año hasta = 2025**.
- c) Haga clic en **Buscar** para cargar los resultados.
- d) De la primera página de resultados, extraiga de cada anuncio: **marca, año, precio**.
- e) Guarde los datos en un **DataFrame** con columnas: **marca, ano, precio colones**. Nota: el nombre de columna ano se escribe sin tilde para evitar problemas de codificación en el
archivo CSV.
- f) Exporte los datos como `crautos_2018_2025.csv`.



[↑ Volver al Menú](#menu)

### Navegar y extraer datos

In [19]:
browser = webdriver.Firefox()
browser.get('https://crautos.com/autosusados/index.cfm')

## Part 2
- b) Seleccione **Año desde = 2018 y Año hasta = 2025**.
- c) Haga clic en **Buscar** para cargar los resultados.

In [20]:
year_from = browser.find_element(By.XPATH, "//select[@name='yearfrom']/option[@selected]")
year_from.send_keys(2018)

In [21]:
year_to = browser.find_element(By.XPATH, "//select[@name='yearto']/option[@selected]")
year_to.send_keys(2025)

In [22]:
search_button = browser.find_element(By.XPATH, "//button[@type='submit']")
search_button.click()

## Part 3
- d) De la primera página de resultados, extraiga de cada anuncio: **marca, año, precio**.
- e) Guarde los datos en un **DataFrame** con columnas: **marca, ano, precio colones**. Nota: el nombre de columna ano se escribe sin tilde para evitar problemas de codificación en el
archivo CSV.

In [23]:
vehicle_container = browser.find_elements(By.XPATH, "//div[contains(@class, 'd-none d-md-block d-lg-block d-xl-block')]")
print(len(vehicle_container))
print(vehicle_container[0])
print(type(vehicle_container[0]))

16
<selenium.webdriver.remote.webelement.WebElement (session="761ea713-b452-4f87-9c68-9b0cb26c3fd6", element="214f5b37-ddfd-40f2-be78-8d93f59997a6")>
<class 'selenium.webdriver.remote.webelement.WebElement'>


In [24]:
vehicle_tables = []

for container in vehicle_container:
    try:
        table = container.find_element(By.TAG_NAME, "table")
        vehicle_tables.append(table)
    except:
        exception = "No se encontro una tabla"

print(len(vehicle_tables))

16


In [25]:
vehicles_data = pd.read_html(StringIO(vehicle_tables[0].get_attribute("outerHTML")))[0]
vehicles_data = vehicles_data.iloc[:, [0, 4]]
vehicles_data.columns = ["marca", "precio"]
vehicles_data.loc[:, "precio"] = vehicles_data["precio"][1]
vehicles_data['precio'] = vehicles_data['precio'].str.replace(r"\s*\(.*\)\*", "", regex=True)
vehicles_data['precio'] = vehicles_data['precio'].str.replace("¢", "").str.replace(",", "")
vehicles_data = vehicles_data[:1]
vehicles_data

,marca,precio
0,Great Wall M4 2019 • 5 Pas.,3500000


In [26]:
vehicles_data['marca'] = vehicles_data['marca'].str.split("•").str[0].str.strip()
vehicles_data

,marca,precio
0,Great Wall M4 2019,3500000


In [27]:
vehicles_data[['marca', 'anno']] = vehicles_data['marca'].str.extract(r'(.+?)\s(\d{4})$')
vehicles_data

,marca,precio,anno
0,Great Wall M4,3500000,2019


In [28]:
browser.quit()

In [29]:
# try:
#     vehicles2018_2025 = []
#     for vehicles in vehicle_tables:
#         data = pd.read_html(StringIO(vehicles.get_attribute("outerHTML")))[0]
#         data = data.iloc[:, [0, 4]]
#         data.columns = ["marca", "precio"]
#         data.loc[:, "precio"] = data["precio"][1]
#         data['precio'] = data['precio'].str.replace(r"\s*\(.*\)\*", "", regex=True)
#         data['precio'] = data['precio'].str.replace("¢", "").str.replace(",", "")
#         data = data[:1]
#         data['marca'] = data['marca'].str.split("•").str[0].str.strip()
#         data[['marca', 'anno']] = data['marca'].str.extract(r'(.+?)\s(\d{4})$')
    
    
#         vehicles2018_2025.append(data)
# except Exception as e:
#     print("ERROR:", e)
    
# finally:
#     pass
#     # browser.quit()

El card trae demasiada información que en este caso no necesitamos, por lo que haciendo pruebas para extrae un solo vehículo es mas complejo trayendo la tabla (con read_html...) y filtrando, xq si no existen algunos datos entonces se va a extraer mal los datos.  
Por lo tanto, vamos a ir un poco más directo en lo que necesitamos de cada card en especifico como la marca (.brandtitle), separando el año de la marca y eliminando los pasajeros y extrayendo el precio (.precio) en colones y limpiando lo para que quede como un integer.

In [30]:
def scrape_vehicles_2018_2025():
    try:
        vehicles2018_2025 = []
        
        # Inicializar navegador
        options = Options()
        options.add_argument("--headless") # no muestra la ventana del navegador
        browser = webdriver.Firefox(options=options)
        browser.get('https://crautos.com/autosusados/index.cfm')
        
        # Espera de hasta 60 segundos
        wait = WebDriverWait(browser, 60)
        wait.until(EC.presence_of_element_located((By.XPATH, "//div[contains(@class, 'searchform')]")))

        # Seleccionar years y buscar haciendo click
        year_from = browser.find_element(By.XPATH, "//select[@name='yearfrom']/option[@selected]")
        year_from.send_keys(2018)
        year_to = browser.find_element(By.XPATH, "//select[@name='yearto']/option[@selected]")
        year_to.send_keys(2025)
        search_button = browser.find_element(By.XPATH, "//button[@type='submit']")
        search_button.click()
        
        # esperamos hasta que se cargue el resultado de la busqueda
        wait.until(EC.presence_of_element_located((By.XPATH, "//form[contains(@name, 'form')]")))
        vehicle_tables = browser.find_elements(By.XPATH, "//div[contains(@class, 'd-none d-md-block d-lg-block d-xl-block')]")
        # print('Resultados encontrados:', len(vehicle_tables))
        
        
        for vehicle in vehicle_tables:
            try:
                # extraemos la marca de cada vehicle
                marca = vehicle.find_element(By.CLASS_NAME, "brandtitle").text
                
                # eliminamos "• 5 Pas."
                marca = marca.split("•")[0].strip()
                
                # y usamos regex para extraer el year (si existe, sino none)
                year = re.search(r"(.+?)\s(\d{4})$", marca)
                
                if year:
                    marca, year = year.groups()
                else:
                    year = None
                
                
                # extraemos el precio de cada vehicle
                precio = vehicle.find_element(By.CLASS_NAME, "precio").text
                
                # limpiamos el precio eliminando cualquier símbolo de colones, comas y espacios
                precio = precio.replace("¢", "")
                precio = precio.replace(",", "")
                precio = precio.strip()
                
                vehicles2018_2025.append({
                    "marca": marca,
                    "anno": int(year) if year else None,
                    "precio colones": int(precio)
                })

            except:
                continue

    except Exception as e:
        print("ERROR:", e)

    finally:
        browser.quit()
        
    return pd.DataFrame(vehicles2018_2025)

In [31]:
crautos_2018_2025 = scrape_vehicles_2018_2025()
crautos_2018_2025

,marca,anno,precio colones
0,Great Wall M4,2019,3500000
1,Renault KWID,2021,3500000
2,Hyundai GRAND I10,2019,1000000
3,BYD SEAGULL,2024,1200000
4,Geely GX3,2020,2652000
5,Geely GX3,2020,3141000
6,BYD F3,2018,3200000
7,Donfeng (ZNA) GLORY 580,2019,3400000
8,Geely GX3,2020,3401000
9,Chevrolet BEAT LT,2018,3450000


## Parte 4
- f) Exporte los datos como `crautos_2018_2025.csv`.

In [32]:
crautos_2018_2025.to_csv("crautos_2018_2025.csv", index=False, encoding="utf-8-sig")